# CIFAR-10 CNN – quick demo: what is in this image?

Upload any photos and the trained CNN guesses what each one shows.

**How to use (Google Colab):**
1. **Runtime → Run all.** No GPU is needed.
2. When the **Choose Files** button appears under the last cell, pick one or more images from your computer.
3. Each image is shown with the CNN's top-3 guesses. Run the last cell again to try more images.

The CNN knows only the 10 CIFAR-10 classes: **airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck**. Anything else will be forced into the closest of these.

Model: 591,850 parameters, **81.1 % test accuracy** on CIFAR-10 (from `cnn_v3.ipynb` in the same repository).

In [ ]:
# Load the trained model from the project repository
import io, os, sys
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image

MODEL_URL = 'https://raw.githubusercontent.com/sinegan-ux/epuck-cnn-part2/main/controllers/opencv/final_model.keras'
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

model_path = tf.keras.utils.get_file('cifar10_cnn_final_model.keras', MODEL_URL)
model = tf.keras.models.load_model(model_path, compile=False)
print("Model loaded:", model.count_params(), "parameters")

In [ ]:
def prepare(img):
    # Centre square crop (keeps the object's shape), then resize to 32x32 like CIFAR-10
    img = img.convert('RGB')
    w, h = img.size
    s = min(w, h)
    img = img.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s))
    return img, np.asarray(img.resize((32, 32), Image.BILINEAR), dtype=np.float32)[None] / 255.0

def classify_and_show(name, data):
    img, x = prepare(Image.open(io.BytesIO(data)))
    probs = model.predict(x, verbose=0)[0]
    top = probs.argsort()[::-1][:3]

    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(9, 3.6), gridspec_kw={'width_ratios': [1, 1.3]})
    ax_img.imshow(img)
    ax_img.set_title(f"{CLASS_NAMES[top[0]]} ({probs[top[0]]:.0%})", fontsize=14, fontweight='bold')
    ax_img.axis('off')
    labels = [CLASS_NAMES[i] for i in top][::-1]
    values = [probs[i] * 100 for i in top][::-1]
    bars = ax_bar.barh(labels, values, color=['#9DB4D9', '#9DB4D9', '#2F5597'])
    ax_bar.bar_label(bars, fmt='%.1f %%', padding=4)
    ax_bar.set_xlim(0, 115)
    ax_bar.set_xlabel('Confidence (%)')
    ax_bar.set_title('Top-3 guesses')
    for side in ('top', 'right'):
        ax_bar.spines[side].set_visible(False)
    fig.suptitle(name, fontsize=9, color='gray')
    plt.tight_layout()
    plt.show()
    print(f"{name}: {', '.join(f'{CLASS_NAMES[i]} {probs[i]:.1%}' for i in top)}")

In [ ]:
# Upload images and classify them (run this cell again to try more)
if 'google.colab' in sys.modules:
    from google.colab import files
    uploaded = files.upload()
else:                                   # local Jupyter: classify the images in my_images/
    folder = 'my_images'
    uploaded = {f: open(os.path.join(folder, f), 'rb').read() for f in sorted(os.listdir(folder))
                if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.webp', '.avif'))}

if not uploaded:
    print("No images selected.")
for name, data in uploaded.items():
    try:
        classify_and_show(name, data)
    except Exception as exc:
        print(f"{name}: could not read this file as an image ({exc})")